In [8]:
from pathlib import Path
import sys
import gc
import time

import numpy as np
import pandas as pd

# Locate project root: the folder containing common.py
cwd = Path.cwd()

if (cwd / "common.py").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "common.py").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "common.py not found. Run this notebook from "
        "the project root or its baseline/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common import (
    LANGS,
    RUN_DIR,
    RES_DIR,
    load_benchmark,
    record,
    rrf_fuse,
)

print("Project root:", PROJECT_ROOT)
print("Runs directory:", RUN_DIR)
print("Results directory:", RES_DIR)
print("Languages:", LANGS)
print("Number of languages:", len(LANGS))

assert len(LANGS) == 13
assert RUN_DIR.exists()

Project root: c:\Users\sanka\OneDrive\Desktop\research paper
Runs directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs
Results directory: C:\Users\sanka\OneDrive\Desktop\research paper\results
Languages: ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'ta', 'te', 'ur']
Number of languages: 13


In [9]:
print("Available run directories:\n")

for folder in sorted(RUN_DIR.iterdir()):
    if not folder.is_dir():
        continue

    score_files = list(folder.glob("*_scores.npy"))
    rank_files = list(folder.glob("*_ranks.npy"))

    print(
        f"{folder.name:25s} "
        f"scores={len(score_files):2d} "
        f"ranks={len(rank_files):2d}"
    )

Available run directories:

bge-m3                    scores=13 ranks=13
bm25                      scores=13 ranks=13
indicbert-v3              scores= 1 ranks= 1
mE5-base                  scores=13 ranks=13
mE5-large                 scores=13 ranks=13
mE5-small                 scores=13 ranks=13
muril                     scores=13 ranks=13


In [10]:
# Change this only if your BM25 directory has another name.
BM25_TAG = "bm25"

BGE_TAG = "bge-m3"

# RRF hyperparameter
RRF_K = 60

# Fuse rankings using all documents in the corpus.
RRF_DEPTH = None

# Model tag used for the hybrid outputs.
HYBRID_TAG = "bm25+bge-m3-rrf"

BM25_DIR = RUN_DIR / BM25_TAG
BGE_DIR = RUN_DIR / BGE_TAG
HYBRID_DIR = RUN_DIR / HYBRID_TAG

HYBRID_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

print("BM25 directory:", BM25_DIR)
print("BGE directory:", BGE_DIR)
print("Hybrid output directory:", HYBRID_DIR)
print("RRF k:", RRF_K)
print("RRF depth:", RRF_DEPTH)

assert BM25_DIR.is_dir(), f"BM25 directory not found: {BM25_DIR}"
assert BGE_DIR.is_dir(), f"BGE directory not found: {BGE_DIR}"

BM25 directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\bm25
BGE directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\bge-m3
Hybrid output directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\bm25+bge-m3-rrf
RRF k: 60
RRF depth: None


In [11]:
validation_errors = []

for lang in LANGS:
    bm25_scores_path = BM25_DIR / f"{lang}_scores.npy"
    bge_scores_path = BGE_DIR / f"{lang}_scores.npy"

    bm25_ranks_path = BM25_DIR / f"{lang}_ranks.npy"
    bge_ranks_path = BGE_DIR / f"{lang}_ranks.npy"

    required_paths = [
        bm25_scores_path,
        bge_scores_path,
        bm25_ranks_path,
        bge_ranks_path,
    ]

    missing = [p.name for p in required_paths if not p.exists()]

    if missing:
        validation_errors.append(
            f"{lang}: missing files {missing}"
        )
        continue

    queries, corpus, gold = load_benchmark(lang)

    bm25_scores = np.load(bm25_scores_path, mmap_mode="r")
    bge_scores = np.load(bge_scores_path, mmap_mode="r")

    bm25_ranks = np.load(bm25_ranks_path, mmap_mode="r")
    bge_ranks = np.load(bge_ranks_path, mmap_mode="r")

    expected_scores_shape = (len(queries), len(corpus))
    expected_ranks_shape = (len(queries),)

    if bm25_scores.shape != expected_scores_shape:
        validation_errors.append(
            f"{lang}: BM25 shape {bm25_scores.shape}, "
            f"expected {expected_scores_shape}"
        )

    if bge_scores.shape != expected_scores_shape:
        validation_errors.append(
            f"{lang}: BGE shape {bge_scores.shape}, "
            f"expected {expected_scores_shape}"
        )

    if bm25_ranks.shape != expected_ranks_shape:
        validation_errors.append(
            f"{lang}: BM25 ranks shape {bm25_ranks.shape}"
        )

    if bge_ranks.shape != expected_ranks_shape:
        validation_errors.append(
            f"{lang}: BGE ranks shape {bge_ranks.shape}"
        )

    if bm25_scores.shape == bge_scores.shape == expected_scores_shape:
        print(
            f"{lang}: queries={len(queries)}, "
            f"documents={len(corpus)}, "
            f"shape={bm25_scores.shape}"
        )

    del bm25_scores, bge_scores, bm25_ranks, bge_ranks

if validation_errors:
    print("\nVALIDATION FAILED:")
    for error in validation_errors:
        print("-", error)

    raise RuntimeError(
        "Fix the input files before running hybrid retrieval."
    )

print("\nPASS: all 13 languages have compatible input files.")

as: queries=1000, documents=1000, shape=(1000, 1000)
bn: queries=1000, documents=1000, shape=(1000, 1000)
gu: queries=999, documents=999, shape=(999, 999)
hi: queries=1000, documents=1000, shape=(1000, 1000)
kn: queries=1000, documents=1000, shape=(1000, 1000)
ml: queries=1000, documents=1000, shape=(1000, 1000)
mr: queries=1000, documents=1000, shape=(1000, 1000)
ne: queries=1000, documents=1000, shape=(1000, 1000)
or: queries=1000, documents=1000, shape=(1000, 1000)
pa: queries=1000, documents=1000, shape=(1000, 1000)
ta: queries=1000, documents=1000, shape=(1000, 1000)
te: queries=1000, documents=1000, shape=(1000, 1000)
ur: queries=1000, documents=1000, shape=(1000, 1000)

PASS: all 13 languages have compatible input files.


In [12]:
lang = "hi"

bm25_scores = np.load(BM25_DIR / f"{lang}_scores.npy")
bge_scores = np.load(BGE_DIR / f"{lang}_scores.npy")

queries, corpus, gold = load_benchmark(lang)

assert bm25_scores.shape == bge_scores.shape
assert bm25_scores.shape == (len(queries), len(corpus))
assert len(gold) == len(queries)

t0 = time.time()

hybrid_scores = rrf_fuse(
    score_mats=[bm25_scores, bge_scores],
    k=RRF_K,
    depth=RRF_DEPTH,
)

elapsed = time.time() - t0

assert hybrid_scores.shape == bm25_scores.shape
assert np.isfinite(hybrid_scores).all()

# Verify the fused ranking can be recorded correctly.
hybrid_metrics = record(
    model=HYBRID_TAG,
    key=lang,
    S=hybrid_scores,
    gold=gold,
    setting="mono",
)

print(f"Language: {lang}")
print(f"Queries: {len(queries)}")
print(f"Documents: {len(corpus)}")
print(f"Hybrid matrix shape: {hybrid_scores.shape}")
print(f"Fusion time: {elapsed:.3f} seconds")
print(f"MRR: {hybrid_metrics['MRR']:.4f}")
print(f"MRR@10: {hybrid_metrics['MRR@10']:.4f}")
print(f"Recall@1: {hybrid_metrics['R@1']:.4f}")
print(f"Recall@5: {hybrid_metrics['R@5']:.4f}")
print(f"Recall@10: {hybrid_metrics['R@10']:.4f}")

Language: hi
Queries: 1000
Documents: 1000
Hybrid matrix shape: (1000, 1000)
Fusion time: 0.252 seconds
MRR: 0.8005
MRR@10: 0.7967
Recall@1: 0.7310
Recall@5: 0.8770
Recall@10: 0.9060


In [13]:
all_metrics = []
failed_languages = []

for lang in LANGS:
    print("\n" + "=" * 65)
    print(f"Running BM25 + BGE-M3 RRF: {lang}")
    print("=" * 65)

    try:
        start = time.time()

        # Load saved baseline score matrices.
        bm25_scores = np.load(
            BM25_DIR / f"{lang}_scores.npy",
            mmap_mode="r",
        )

        bge_scores = np.load(
            BGE_DIR / f"{lang}_scores.npy",
            mmap_mode="r",
        )

        # Validate against the canonical benchmark.
        queries, corpus, gold = load_benchmark(lang)

        expected_shape = (len(queries), len(corpus))

        assert bm25_scores.shape == expected_shape
        assert bge_scores.shape == expected_shape
        assert len(gold) == len(queries)

        # Reciprocal Rank Fusion.
        hybrid_scores = rrf_fuse(
            score_mats=[bm25_scores, bge_scores],
            k=RRF_K,
            depth=RRF_DEPTH,
        )

        assert hybrid_scores.shape == expected_shape
        assert np.isfinite(hybrid_scores).all()

        # Save fused scores, ranks and metrics.
        metrics = record(
            model=HYBRID_TAG,
            key=lang,
            S=hybrid_scores,
            gold=gold,
            setting="mono",
        )

        all_metrics.append(metrics)

        print(f"MRR:       {metrics['MRR']:.4f}")
        print(f"MRR@10:    {metrics['MRR@10']:.4f}")
        print(f"Recall@1:  {metrics['R@1']:.4f}")
        print(f"Recall@5:  {metrics['R@5']:.4f}")
        print(f"Recall@10: {metrics['R@10']:.4f}")
        print(f"Recall@20: {metrics['R@20']:.4f}")
        print(f"Recall@100:{metrics['R@100']:.4f}")
        print(f"Time: {(time.time() - start):.2f} seconds")

        del bm25_scores, bge_scores, hybrid_scores
        gc.collect()

    except Exception as exc:
        print(f"FAILED: {lang}")
        print(f"Error: {exc}")
        failed_languages.append(lang)
        gc.collect()

print("\n" + "=" * 65)
print("HYBRID RRF RUN FINISHED")
print("=" * 65)
print("Successful this session:", len(all_metrics))
print("Failed languages:", failed_languages or "None")


Running BM25 + BGE-M3 RRF: as
MRR:       0.6094
MRR@10:    0.5975
Recall@1:  0.5290
Recall@5:  0.6830
Recall@10: 0.7440
Recall@20: 0.8540
Recall@100:0.9710
Time: 0.33 seconds

Running BM25 + BGE-M3 RRF: bn
MRR:       0.7397
MRR@10:    0.7337
Recall@1:  0.6650
Recall@5:  0.8240
Recall@10: 0.8590
Recall@20: 0.9130
Recall@100:0.9820
Time: 0.27 seconds

Running BM25 + BGE-M3 RRF: gu
MRR:       0.6752
MRR@10:    0.6671
Recall@1:  0.5926
Recall@5:  0.7588
Recall@10: 0.8128
Recall@20: 0.8919
Recall@100:0.9780
Time: 0.28 seconds

Running BM25 + BGE-M3 RRF: hi
MRR:       0.8005
MRR@10:    0.7967
Recall@1:  0.7310
Recall@5:  0.8770
Recall@10: 0.9060
Recall@20: 0.9410
Recall@100:0.9860
Time: 0.30 seconds

Running BM25 + BGE-M3 RRF: kn
MRR:       0.6542
MRR@10:    0.6426
Recall@1:  0.5860
Recall@5:  0.7090
Recall@10: 0.7620
Recall@20: 0.8890
Recall@100:0.9710
Time: 0.24 seconds

Running BM25 + BGE-M3 RRF: ml
MRR:       0.6515
MRR@10:    0.6418
Recall@1:  0.5770
Recall@5:  0.7230
Recall@10: 0.7850

In [14]:
verification_errors = []

for lang in LANGS:
    scores_path = HYBRID_DIR / f"{lang}_scores.npy"
    ranks_path = HYBRID_DIR / f"{lang}_ranks.npy"

    if not scores_path.exists():
        verification_errors.append(f"{lang}: missing scores file")
        continue

    if not ranks_path.exists():
        verification_errors.append(f"{lang}: missing ranks file")
        continue

    queries, corpus, gold = load_benchmark(lang)

    scores = np.load(scores_path, mmap_mode="r")
    ranks = np.load(ranks_path)

    if scores.shape != (len(queries), len(corpus)):
        verification_errors.append(
            f"{lang}: incorrect score shape {scores.shape}"
        )

    if ranks.shape != (len(queries),):
        verification_errors.append(
            f"{lang}: incorrect rank shape {ranks.shape}"
        )

    if not np.isfinite(scores).all():
        verification_errors.append(
            f"{lang}: non-finite scores"
        )

    if np.any(ranks < 1) or np.any(ranks > len(corpus)):
        verification_errors.append(
            f"{lang}: invalid gold-document ranks"
        )

    print(
        f"{lang}: scores={scores.shape}, "
        f"ranks={ranks.shape}, verified"
    )

print("\nScore files:",
      sum((HYBRID_DIR / f"{l}_scores.npy").exists() for l in LANGS),
      "/ 13")

print("Rank files:",
      sum((HYBRID_DIR / f"{l}_ranks.npy").exists() for l in LANGS),
      "/ 13")

if verification_errors:
    for error in verification_errors:
        print("ERROR:", error)
    raise RuntimeError("Output verification failed.")

print("\nPASS: all hybrid output files passed verification.")

as: scores=(1000, 1000), ranks=(1000,), verified
bn: scores=(1000, 1000), ranks=(1000,), verified
gu: scores=(999, 999), ranks=(999,), verified
hi: scores=(1000, 1000), ranks=(1000,), verified
kn: scores=(1000, 1000), ranks=(1000,), verified
ml: scores=(1000, 1000), ranks=(1000,), verified
mr: scores=(1000, 1000), ranks=(1000,), verified
ne: scores=(1000, 1000), ranks=(1000,), verified
or: scores=(1000, 1000), ranks=(1000,), verified
pa: scores=(1000, 1000), ranks=(1000,), verified
ta: scores=(1000, 1000), ranks=(1000,), verified
te: scores=(1000, 1000), ranks=(1000,), verified
ur: scores=(1000, 1000), ranks=(1000,), verified

Score files: 13 / 13
Rank files: 13 / 13

PASS: all hybrid output files passed verification.


In [15]:
metrics_path = RES_DIR / "metrics.csv"

assert metrics_path.exists(), "metrics.csv not found."

metrics_df = pd.read_csv(metrics_path)

required_models = [BM25_TAG, BGE_TAG, HYBRID_TAG]

comparison = metrics_df[
    (metrics_df["model"].isin(required_models))
    & (metrics_df["setting"] == "mono")
].copy()

metric_columns = [
    "MRR",
    "MRR@10",
    "R@1",
    "R@5",
    "R@10",
    "R@20",
    "R@100",
]

comparison = comparison[
    comparison["key"].isin(LANGS)
]

# Keep one row per model and language.
comparison = comparison.drop_duplicates(
    subset=["model", "key", "setting"],
    keep="last",
)

for model_name in required_models:
    present = set(
        comparison.loc[
            comparison["model"] == model_name, "key"
        ]
    )
    missing = set(LANGS) - present

    print(f"{model_name}: {len(present)}/13 languages")
    if missing:
        print("  Missing:", sorted(missing))

# Pivot MRR for direct per-language comparison.
mrr_comparison = comparison.pivot(
    index="key",
    columns="model",
    values="MRR",
).reindex(LANGS)

print("\nMRR comparison:")
display(mrr_comparison.round(4))

# Pivot Recall@10.
r10_comparison = comparison.pivot(
    index="key",
    columns="model",
    values="R@10",
).reindex(LANGS)

print("\nRecall@10 comparison:")
display(r10_comparison.round(4))

bm25: 13/13 languages
bge-m3: 13/13 languages
bm25+bge-m3-rrf: 13/13 languages

MRR comparison:


model,bge-m3,bm25,bm25+bge-m3-rrf
key,,,
as,0.8073,0.4616,0.6094
bn,0.8602,0.6189,0.7397
gu,0.8336,0.5465,0.6752
hi,0.8863,0.7117,0.8005
kn,0.8379,0.5380,0.6542
ml,0.8405,0.5248,0.6515
mr,0.8521,0.5097,0.6683
ne,0.8504,0.5052,0.6557
or,0.7953,0.5172,0.6477



Recall@10 comparison:


model,bge-m3,bm25,bm25+bge-m3-rrf
key,,,
as,0.9110,0.5990,0.7440
bn,0.9430,0.7720,0.8590
gu,0.9369,0.6877,0.8128
hi,0.9570,0.8420,0.9060
kn,0.9200,0.6630,0.7620
ml,0.9270,0.6620,0.7850
mr,0.9390,0.6630,0.7850
ne,0.9440,0.6610,0.8090
or,0.9050,0.6520,0.7830


In [16]:
if not all(
    model_name in mrr_comparison.columns
    for model_name in required_models
):
    raise RuntimeError(
        "Cannot compare all three systems. "
        "Check the missing model rows in Cell 8."
    )

delta = pd.DataFrame(index=LANGS)

delta["BM25_MRR"] = mrr_comparison[BM25_TAG]
delta["BGE_MRR"] = mrr_comparison[BGE_TAG]
delta["Hybrid_MRR"] = mrr_comparison[HYBRID_TAG]

delta["Hybrid_minus_BM25"] = (
    delta["Hybrid_MRR"] - delta["BM25_MRR"]
)

delta["Hybrid_minus_BGE"] = (
    delta["Hybrid_MRR"] - delta["BGE_MRR"]
)

delta["Best_baseline_MRR"] = delta[
    ["BM25_MRR", "BGE_MRR"]
].max(axis=1)

delta["Hybrid_beats_best_baseline"] = (
    delta["Hybrid_MRR"] > delta["Best_baseline_MRR"]
)

display(delta.round(4))

print(
    "Languages where hybrid beats both baselines:",
    int(delta["Hybrid_beats_best_baseline"].sum()),
    "/ 13",
)

print("\nMacro-average MRR:")
print(delta[
    ["BM25_MRR", "BGE_MRR", "Hybrid_MRR"]
].mean().round(4))

,BM25_MRR,BGE_MRR,Hybrid_MRR,Hybrid_minus_BM25,Hybrid_minus_BGE,Best_baseline_MRR,Hybrid_beats_best_baseline
as,0.4616,0.8073,0.6094,0.1477,-0.1979,0.8073,False
bn,0.6189,0.8602,0.7397,0.1208,-0.1205,0.8602,False
gu,0.5465,0.8336,0.6752,0.1286,-0.1584,0.8336,False
hi,0.7117,0.8863,0.8005,0.0888,-0.0858,0.8863,False
kn,0.5380,0.8379,0.6542,0.1162,-0.1837,0.8379,False
ml,0.5248,0.8405,0.6515,0.1266,-0.1890,0.8405,False
mr,0.5097,0.8521,0.6683,0.1586,-0.1838,0.8521,False
ne,0.5052,0.8504,0.6557,0.1505,-0.1948,0.8504,False
or,0.5172,0.7953,0.6477,0.1305,-0.1476,0.7953,False
pa,0.6532,0.8298,0.7530,0.0998,-0.0769,0.8298,False


Languages where hybrid beats both baselines: 0 / 13

Macro-average MRR:
BM25_MRR      0.5651
BGE_MRR       0.8384
Hybrid_MRR    0.6930
dtype: float64
